In [ ]:
%pip -q install openai pandas numpy scipy tqdm pillow

from google.colab import drive, userdata
from IPython.display import display
from pathlib import Path
from datetime import datetime, timezone
import base64
import json
import os
import re
import time

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from openai import OpenAI

In [ ]:
METRIC = 'CC'
TARGET_COLUMN = 'CRAI_CC'
MODEL = 'gpt-5.4'
REASONING_EFFORT = 'none'
IMAGE_DETAIL = 'original'
MAX_RETRIES = 3
REQUEST_SLEEP_SECONDS = 0.20

                                                   
RUN_DEV_API_CALLS = True
RUN_TEST_API_CALLS = True

if IN_COLAB:
    drive.mount('/content/drive')
    IMAGEEVAL_ROOT = Path('/content/drive/MyDrive/Dr. Lulwah - Ahmed/ImageEVAl')
else:
    IMAGEEVAL_ROOT = Path(os.environ.get('IMAGEEVAL_ROOT', '.')).resolve()

PROJECT_DIR = IMAGEEVAL_ROOT / 'ImageEval2026_Task2_CRAI_Bench'
DATA_CANDIDATES = [PROJECT_DIR / 'data', IMAGEEVAL_ROOT / 'train_dev']
DATA_DIR = next(
    (path for path in DATA_CANDIDATES
     if (path / 'dev' / 'captions.tsv').exists()),
    DATA_CANDIDATES[0],
)

EXPERIMENT_ROOT = PROJECT_DIR / 'cc_direct_gpt54_no_training_v1'
CACHE_DIR = EXPERIMENT_ROOT / 'cache'
OUTPUT_DIR = EXPERIMENT_ROOT / 'outputs'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def parse_base_id(instance_id):
    return re.sub(r'_v\d+$', '', str(instance_id))

def parse_version(instance_id):
    match = re.search(r'_v(\d+)$', str(instance_id))
    return int(match.group(1))

def find_image(folder, stem):
    for suffix in ['.png', '.jpg', '.jpeg', '.webp']:
        path = folder / f'{stem}{suffix}'
        if path.exists():
            return str(path)

def load_inputs(split):
    folder = DATA_DIR / split
    frame = pd.read_csv(folder / 'captions.tsv', sep='\t')
    frame['id'] = frame['id'].astype(str)
    frame['base_id'] = frame['id'].map(parse_base_id)
    frame['caption_version'] = frame['id'].map(parse_version)
    frame['ref_image_path'] = frame['base_id'].map(
        lambda value: find_image(folder / 'imgs' / 'ref', value)
    )
    frame['generated_image_path'] = frame['id'].map(
        lambda value: find_image(folder / 'imgs' / 'generated', value)
    )
    return frame

dev_df = load_inputs('dev')
test_df = load_inputs('test') if (DATA_DIR / 'test' / 'captions.tsv').exists() else pd.DataFrame()

In [ ]:
PROMPT_VERSION = 'cc-direct-no-training-gpt54-v1'

SYSTEM_PROMPT = 'ROLE\n\nYou are a strict multimodal evaluator for ImageEval 2026 CRAI-Bench. Judge only\nCRAI_CC: Contextual Coherence.\n\nTARGET\n\nJudge whether the visible cultural elements form a coherent and plausible scene. Consider\nthe setting, actions, relationships, placement, and function of the elements. A high score\nrequires the elements to fit together naturally in the intended cultural context. A low\nscore is appropriate when otherwise recognizable elements appear in an incompatible,\nimplausible, or functionally wrong setting.\n\nDo not turn this into CEA (mere element presence), CS (how uniquely Qatari the image is),\nCI (cultural identity or distortion), HP (unsupported fabricated additions), or an image\nquality score. Use the current caption to determine the requested scene and the authentic\nreference image to understand its intended context.\n\nSCORE BANDS\n\n- 0.00--0.20: the scene is culturally or functionally incoherent.\n- 0.25--0.45: major contextual conflicts dominate.\n- 0.50--0.65: partly coherent, with noticeable contextual problems.\n- 0.70--0.85: mostly coherent; only minor contextual problems remain.\n- 0.90--1.00: the setting, relations, and actions are fully coherent.\n\nGROUP PROCEDURE\n\nYou will receive one authentic reference image and five independently generated variants,\neach paired with its current caption. Score every variant independently. Do not assume a\nfixed score distribution and do not compare the variants merely to rank them.\n\nDIRECT-SCORING REQUIREMENT\n\nMake one holistic judgment for each target. Do not generate statements, questions, cultural\nanchors, submetric scores, or a feature vector. The numeric score is the prediction.\n\nOUTPUT JSON\n\n{\n  "items": [\n    {\n      "id": "TARGET_INSTANCE_ID",\n      "score": 0.0,\n      "confidence": 0.0,\n      "visible_evidence": "one short visible observation",\n      "brief_reason": "one short sentence"\n    }\n  ]\n}\n\nReturn JSON only with exactly one item per target variant. Scores and confidence must be\nbetween 0 and 1. Do not output chain-of-thought and do not score any other CRAI metric.'.strip()

In [ ]:
def image_to_data_url(path):
    path = Path(path)
    mime = {'.png': 'image/png', '.jpg': 'image/jpeg',
            '.jpeg': 'image/jpeg', '.webp': 'image/webp'}[path.suffix.lower()]
    encoded = base64.b64encode(path.read_bytes()).decode('utf-8')
    return f'data:{mime};base64,{encoded}'

def parse_response(text):
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip())
    value = json.loads(text[text.find('{'):text.rfind('}') + 1])
    return value['items']

def cache_path(split):
    return CACHE_DIR / f'{METRIC.lower()}_{split}_direct.jsonl'

def load_jsonl(path):
    if not path.exists():
        return []
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

def append_jsonl(path, record):
    with path.open('a') as handle:
        handle.write(json.dumps(record) + '\n')

client = None

def get_client():
    global client
    if client is None:
        client = OpenAI(api_key=userdata.get('openai'))
    return client

In [ ]:
def group_content(group):
    group = group.sort_values('caption_version')
    content = [
        {'type': 'input_text', 'text': 'AUTHENTIC REFERENCE IMAGE:'},
        {'type': 'input_image', 'image_url': image_to_data_url(group.iloc[0]['ref_image_path']),
         'detail': IMAGE_DETAIL},
    ]
    for _, row in group.iterrows():
        content.extend([
            {'type': 'input_text',
             'text': f"TARGET {row['id']} | CURRENT CAPTION:\n{row['caption']}"},
            {'type': 'input_image', 'image_url': image_to_data_url(row['generated_image_path']),
             'detail': IMAGE_DETAIL},
        ])
    return content

def call_group(group, split):
    expected_ids = group.sort_values('caption_version')['id'].tolist()
    for attempt in range(MAX_RETRIES):
        try:
            response = get_client().responses.create(
                model=MODEL,
                reasoning={'effort': REASONING_EFFORT},
                input=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': group_content(group)},
                ],
            )
            items = parse_response(response.output_text)
            by_id = {str(item['id']): item for item in items}
            return {
                'base_id': str(group.iloc[0]['base_id']),
                'items': [by_id[instance_id] for instance_id in expected_ids],
                'created_utc': datetime.now(timezone.utc).isoformat(),
            }
        except Exception:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(2 ** (attempt + 1))

def load_or_infer(frame, split, run_calls):
    path = cache_path(split)
    cached = {record['base_id']: record for record in load_jsonl(path)}
    for base_id, group in tqdm(frame.groupby('base_id'), desc=f'{METRIC} {split}'):
        if base_id not in cached and run_calls:
            cached[base_id] = call_group(group, split)
            append_jsonl(path, cached[base_id])
            time.sleep(REQUEST_SLEEP_SECONDS)
    return [cached[key] for key in sorted(cached) if key in set(frame['base_id'])]

def records_to_frame(records):
    rows = []
    for record in records:
        for item in record['items']:
            rows.append({
                'id': str(item['id']),
                TARGET_COLUMN: float(item['score']),
                'confidence': float(item.get('confidence', 0)),
                'visible_evidence': item.get('visible_evidence', ''),
                'brief_reason': item.get('brief_reason', ''),
            })
    return pd.DataFrame(rows)

In [ ]:
dev_records = load_or_infer(dev_df, 'dev', RUN_DEV_API_CALLS)
dev_predictions = records_to_frame(dev_records)

if len(dev_predictions) == len(dev_df):
    gold = pd.read_csv(DATA_DIR / 'dev' / 'gold_human.tsv', sep='\t')
    evaluation = gold[['id', TARGET_COLUMN]].merge(
        dev_predictions[['id', TARGET_COLUMN]], on='id', suffixes=('_gold', '_prediction')
    )
    correlation = spearmanr(
        evaluation[f'{TARGET_COLUMN}_gold'], evaluation[f'{TARGET_COLUMN}_prediction']
    ).statistic
    mae = np.mean(np.abs(
        evaluation[f'{TARGET_COLUMN}_gold'] - evaluation[f'{TARGET_COLUMN}_prediction']
    ))
    display(pd.DataFrame([{'spearman': correlation, 'mae': mae}]).round(4))
    dev_predictions.to_csv(
        OUTPUT_DIR / f'{METRIC.lower()}_dev_direct_predictions.tsv', sep='\t', index=False
    )

if len(test_df):
    test_records = load_or_infer(test_df, 'test', RUN_TEST_API_CALLS)
    test_predictions = records_to_frame(test_records)
    if len(test_predictions) == len(test_df):
        test_predictions[['id', TARGET_COLUMN]].to_csv(
            OUTPUT_DIR / f'test_{METRIC.lower()}_direct_predictions.tsv',
            sep='\t', index=False
        )